# EDA - Exploratory Data Analysis

In this step, before I'll move to Valdiation process, it's necessary understand the structure and characteristics of the datasets. The objective is to identify potential anomalies, inconsistencies, and structural differences between the files.

This step include:
- dataset shape check - to verify if both files have same rows numbers, if not - differences in the number of records may indicate potential discrepancies between the systems.
- data type overview - reviewing the data types assigned to each column in order to identify potential inconsistencies (e.g., numerical values stored as strings or incorrect date formats).
- missing value identification - detecting null values
- unique values analysis - analyzing the number of unique values in key columns (e.g. order identifiers). Repeated values may indicate duplicate records or non-unique keys.
- descriptive statistics - generating summary statistics for numerical variables (mean, average, median, min, max, etc.).

! The purpose of EDA is to explore and understand the data, identify irregularities, and detect potential data quality issues before further processing.

In [ ]:
import pandas as pd 

#data load
def load_data(file_path):
    try:
        df = pd.read_csv(file_path, encoding='latin1')
        print(f"File {file_path} loaded correctly using latin1 encoding. Loaded {len(df)} rows.")
        return df
    except Exception:
        df = pd.read_csv(file_path, encoding='cp1252')
        print(f"File {file_path} loaded correctly using cp1252 encoding. Loaded {len(df)} rows.")
        return df
    
df_oms = load_data('oms_orders_placed.csv')
df_wms = load_data('wms_shipped_units.csv')

File oms_orders_placed.csv loaded correctly using latin1 encoding. Loaded 541909 rows.
File wms_shipped_units.csv loaded correctly using latin1 encoding. Loaded 542409 rows.


Observation: Data loaded correctly, but disrepancy between both files is visible.

**Step 1**
 Initial check of the first 5 rows to verify correct loading and column alignment.

In [15]:
print("OMS Preview:")
display(df_oms.head(5))

print("WMS Preview:")
display(df_wms.head(5))

OMS Preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


WMS Preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,ERR_DATE_2024,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,ERR_DATE_2024,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,ERR_DATE_2024,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,ERR_DATE_2024,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,ERR_DATE_2024,3.39,17850.0,United Kingdom


Observation: 
- InvoiceDate in WMS dataset contains corrupted strings instead of valid timestamps.
- CustomerIS is stored as float.

**Step 2** 
Data shape checking - comparing the number of records betwwen both systems.

In [ ]:
# dataset shape check
print(f"OMS (Orders Placed) rows: {df_oms.shape[0]}, columns: {df_oms.shape[1]}")
print(f"WMS (Shipped Units) rows: {df_wms.shape[0]}, columns: {df_wms.shape[1]}")

if df_oms.shape[0] != df_wms.shape[0]:
    print(f" Discrepancy detected! Difference: {abs(df_oms.shape[0] - df_wms.shape[0])} records.")

OMS (Orders Placed) rows: 541909, columns: 8
WMS (Shipped Units) rows: 542409, columns: 8
 Discrepancy detected! Difference: 500 records.


Observation: WMS dataset has 500 records more.

**Step 3**
Data type overiew - reviewing the data types assigned to each column.

In [21]:
df_oms.info()

df_wms.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542409 entries, 0 to 542408
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    542409 non-null  object 
 1   StockCode    542409 non-null  object 
 2   Description  540955 non-null  object 
 3   Quantity     542409 non-null  int64  
 4   InvoiceDate  542409 non-

Observation: 
- In OMS CustomerID is stored as float64, and in WMS as object.
- In both datasets, the date is stored as an object (string) instead of a datetime object.
- The WMS file has more object type columns than the OMS file, which suggests potential "corrupted" data or different formatting standards in the warehouse system.
- Since CustomerID in OMS dataset is float64, and in WMS changed into object, it can mean that there are hidden spaces.

**Step 4** Missing value identification

In [47]:
print("Missing values in OMS dataset:")
print(df_oms.isnull().sum())

print("\n Missing values in WMS dataset:")
print(df_wms.isnull().sum())

Missing values in OMS dataset:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

 Missing values in WMS dataset:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice        1000
CustomerID     134660
Country             0
dtype: int64


Observation:
- Both files are missing customer IDs for approximately 25% of orders.
- There is no shortage of prices in OMS, but exactly 1000 of them are missing in WMS dataset.
- Both files are missing Description.
- There's no missing InvoiceNo in both files. Since there are missing CustomerID values, it will not be possible to conduct reconciliation process using this value, but in this case I'll use InvoiceNo. 



**Step 5** Unique values identification in key columns

In [44]:
print(f"Unique Invoice Numbers (OMS): {df_oms['InvoiceNo'].nunique()}")
print(f"Unique Invoice Numbers (WMS): {df_wms['InvoiceNo'].nunique()}")


print(f"Duplicate rows in OMS: {df_oms.duplicated().sum()}")
print(f"Duplicate rows in WMS: {df_wms.duplicated().sum()}")


Unique Invoice Numbers (OMS): 25900
Unique Invoice Numbers (WMS): 25900
Duplicate rows in OMS: 5268
Duplicate rows in WMS: 5702


Observation: 
- Both datasets track the same 25 900 unique invoices, which is strong foundation for reconciliation. 
- The OMS dataset contains 5268 duplicates, what means that the source of data is not perfectly unique itself and requires handling during validation.
- The WMS dataset shows 5702 duplicates, which is 434 more than OMS.
- This disrepancy confirms that WMS data has been corrupted. The reconciliation will need to pinpoint exactly which InvoiceNo values are over-reported in WMS.

**Step 6** Descriptive Statistics - generating summary statistics for numerical variables

In [49]:
print("OMS Numerical Summary")
display(df_oms[['Quantity', 'UnitPrice']].describe())

print("\n WMS Numerical Summary")
display(df_wms[['Quantity', 'UnitPrice']].describe())

OMS Numerical Summary


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000



 WMS Numerical Summary


,Quantity,UnitPrice
count,542409.000000,541409.000000
mean,9.553037,4.612698
std,217.982168,96.805714
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


Observations:
- WMS has 542,409 records vs. 541,909 in OMS (+500 records). This confirms the presence of duplicated entries that will cause "over-shipping" errors in the final report.
- Count of unit price in WMS has only 541,409 records, meaning 500 values are missing (nulls). This will result in undervalued total order amounts unless addressed.
- The mean UnitPrice in WMS (4.6127) is slightly higher than in OMS (4.6111). This confirms that a portion of the dataset has been modified (the +0.01 price manipulation), which will lead to financial discrepancies during the validation phase.
- The Maximum Quantity (80,995) and Minimum Quantity (-80,995) are consistent across both datasets.
*Important!** The presence of negative values confirms that the dataset includes returns. Since the maximum value is exceptionally high (80,995), any duplication of these specific records will lead to massive volume discrepancies.

# General Conclusions

- The WMS dataset shows significant discrepancies compared to the OMS source. Initial analysis confirmed structural issues with data types (CustomerID), missing values (UnitPrice), and unexplained shifts in numerical data.
- A subtle shift in the mean price and 500 missing values in UnitPrice were detected. This indicates that a simple total sum comparison will be inaccurate and requires a detailed row-by-row price audit to identify the specific items causing the variance.
- The additional 500 records in WMS point towards a duplication issue during data transmission.
